# 04 — Modelagem (M1/M2/M3)

Três modelos separados com split temporal (treino ≤2015, validação 2016–2019,
teste ≥2020): **M1** expectativa de vida, **M2** mortalidade <5 (regressão) e
**M3** marco alto (classificação). Cada alvo tem um baseline linear/ridge e um
XGBoost (`tree_method="hist"`, NaN nativo, early stopping na validação).

In [1]:
import sys; sys.path.insert(0, ".")
import json
import joblib
import pandas as pd
import plotly.express as px
from src.modeling import dataset

resumo = json.load(open("models/all_eval.json"))
len(resumo)

3

## Comparativo XGBoost vs baseline linear (conjunto de teste)

In [2]:
linhas = []
for r in resumo:
    x = r["test"]; l = r["test_linear"]
    if r["task"] == "regression":
        linhas.append({"modelo": r["model"], "metrica": "RMSE",
                       "XGBoost": x["rmse"], "Linear/Ridge": l["rmse"]})
        linhas.append({"modelo": r["model"], "metrica": "R²",
                       "XGBoost": x["r2"], "Linear/Ridge": l["r2"]})
    else:
        linhas.append({"modelo": r["model"], "metrica": "AUC",
                       "XGBoost": x["auc"], "Linear/Ridge": l["auc"]})
        linhas.append({"modelo": r["model"], "metrica": "F1",
                       "XGBoost": x["f1"], "Linear/Ridge": l["f1"]})
pd.DataFrame(linhas)

,modelo,metrica,XGBoost,Linear/Ridge
0,life_expectancy_reg,RMSE,2.544572,3.253890
1,life_expectancy_reg,R²,0.887029,0.815268
2,child_mortality_reg,RMSE,14.324072,17.507548
3,child_mortality_reg,R²,0.779482,0.670571
4,milestone_high_clf,AUC,0.993821,0.980493
5,milestone_high_clf,F1,0.911565,0.846906


### Leitura

O XGBoost supera o baseline linear em todos os alvos, com vantagem maior em
mortalidade <5 (relação não linear com saneamento/vacinação). O M3 atinge AUC≈0.99,
mas atenção: o rótulo é derivado de `uhc_index`/LE, que **não** entram como features
(evita leakage trivial).

## Importância das variáveis (SHAP, top-10)

In [3]:
from IPython.display import display
for r in resumo:
    info = json.load(open(f"models/{r['model']}_eval.json"))
    imp = pd.DataFrame(info["xgboost"]["importancias"]).head(10)[::-1]
    fig = px.bar(imp, x="importancia", y="feature", orientation="h",
                 title=f"{r['model']} — top features (|SHAP| medio)")
    display(fig)

## M1 — predito vs observado (teste ≥2020)

In [4]:
ds = dataset.build_dataset("life_expectancy", "regression")
X_te, y_te = ds["test"]
modelo = joblib.load("models/life_expectancy_reg_xgb.joblib")
pred = modelo.predict(X_te)
d = pd.DataFrame({"observado": y_te.values, "predito": pred})
fig = px.scatter(d, x="observado", y="predito", opacity=0.4,
                 title="M1: expectativa de vida — teste (≥2020)")
fig.add_shape(type="line", x0=40, y0=40, x1=90, y1=90, line=dict(dash="dash"))
fig

## Conclusões

- Os 3 modelos são utilizáveis: M1 (R²≈0.89), M2 (R²≈0.78), M3 (AUC≈0.99 no teste).
- O ganho do XGBoost sobre o baseline é consistente, mas parcialmente explicado por
  não linearidades e tratamento nativo de missing (dados escassos em países pobres).
- Ressalva pós-COVID: o teste ≥2020 é um regime fora da amostra de treino; o drift
  é monitorado em F11.